# TF-IDF et décisions de taux

On teste un truc simple : est-ce que le texte aide à retrouver la décision courante, ou la décision suivante ?

In [ ]:
import os
import re
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fomc_nlp_matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, f1_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "fomc_documents_raw.csv").exists():
    ROOT = ROOT.parent

raw = pd.read_csv(ROOT / "data" / "raw" / "fomc_documents_raw.csv", parse_dates=["date"])

TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?", re.IGNORECASE)


def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace(" ", " ")
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def count_words(text):
    return len(TOKEN_RE.findall(clean_text(text)))


RATE_PATTERNS = {
    "hike": [
        r"\b(decided|voted|agreed|approved)\b.{0,80}\b(raise|raising|increase|increasing)\b.{0,80}\b(target|federal funds|discount rate)\b",
        r"\b(raise|raising|increase|increasing|increased)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
    ],
    "cut": [
        r"\b(decided|voted|agreed|approved)\b.{0,80}\b(lower|lowering|reduce|reducing|decrease|decreasing|cut)\b.{0,80}\b(target|federal funds|discount rate)\b",
        r"\b(lower|lowering|reduce|reducing|decrease|decreasing|cut)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
    ],
    "hold": [
        r"\b(decided|voted|agreed)\b.{0,80}\b(maintain|keep|leave)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(will|would|shall|to)\b.{0,20}\b(maintain|keep|leave)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(maintain|maintaining|keep|keeping|kept|leave|leaving)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(target range|target for the federal funds rate|federal funds rate)\b.{0,80}\b(unchanged|maintained)\b",
    ],
}


def infer_rate_decision(text):
    text = clean_text(text)
    for label in ["hike", "cut", "hold"]:
        if any(re.search(pattern, text, flags=re.DOTALL) for pattern in RATE_PATTERNS[label]):
            return label
    return "unknown"


df = (
    raw.sort_values("date")
    .reset_index(drop=True)
    .assign(
        statement_clean_text=lambda x: x["statement_text"].map(clean_text),
        minutes_clean_text=lambda x: x["minutes_text"].map(clean_text),
        statement_n_words=lambda x: x["statement_text"].map(count_words),
        minutes_n_words=lambda x: x["minutes_text"].map(count_words),
        rate_decision=lambda x: x["statement_text"].map(infer_rate_decision),
    )
    .assign(next_rate_decision=lambda x: x["rate_decision"].shift(-1))
)

df[["date", "statement_n_words", "minutes_n_words", "rate_decision", "next_rate_decision"]].head()

Deux versions du texte : complet, puis sans les phrases qui annoncent directement la décision de taux.

In [ ]:
DECISION_SENTENCE_RE = re.compile(
    r"decided to|voted to|agreed to|target range|federal funds rate|discount rate|left unchanged|raise|lower|maintain",
    flags=re.IGNORECASE,
)

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def mask_rate_sentences(text):
    sentences = SENTENCE_SPLIT_RE.split("" if pd.isna(text) else str(text))
    kept = [sentence for sentence in sentences if not DECISION_SENTENCE_RE.search(sentence)]
    return clean_text(" ".join(kept))


df = df.assign(
    statement_masked_text=lambda x: x["statement_text"].map(mask_rate_sentences),
    minutes_masked_text=lambda x: x["minutes_text"].map(mask_rate_sentences),
)

df[["date", "statement_clean_text", "statement_masked_text"]].head(2)

Split chronologique : le début sert à entraîner, la fin sert à évaluer.

In [ ]:
LABELS = ["cut", "hold", "hike"]
FED_STOP_WORDS = sorted(ENGLISH_STOP_WORDS | {
    "accessibility", "action", "adriana", "alan", "april", "august", "barkin", "barr", "bernanke",
    "bies", "board", "button", "chair", "chairman", "committee", "cook", "date",
    "connected", "daly", "december", "discount", "donald", "download", "edward", "february",
    "ferguson", "federal", "fomc", "geithner", "gov", "governors", "gramlich", "greenspan", "home", "https", "immediate",
    "information", "janet", "jefferson", "jerome", "john", "jr", "kugler", "lock",
    "january", "july", "june", "kohn", "lisa", "louis", "main", "march", "mark", "market", "mary",
    "may", "meeting", "menu", "messrs", "michael", "monetary", "mr", "ms", "november",
    "october", "official", "olson", "open", "page", "philip", "policy", "powell", "press",
    "minneapolis", "raphael", "release", "reserve", "roger", "search", "secure", "september", "share", "states",
    "submit", "subscribe", "susan", "timothy", "today", "toggle", "united", "update", "website", "websites",
    "william", "yellen",
})

MODELS = {
    "majority": DummyClassifier(strategy="most_frequent"),
    "logreg": LogisticRegression(max_iter=2_000, class_weight="balanced"),
    "linear_svm": LinearSVC(class_weight="balanced", max_iter=5_000),
}


def fit_text_model(data, text_col, target_col, model_name, estimator, test_size=0.30):
    sample = (
        data.dropna(subset=[target_col])
        .loc[lambda x: x[target_col].isin(LABELS)]
        .sort_values("date")
        .reset_index(drop=True)
    )
    split = int(len(sample) * (1 - test_size))
    train = sample.iloc[:split]
    test = sample.iloc[split:]

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=FED_STOP_WORDS,
            token_pattern=r"(?u)\b[a-z][a-z]+\b",
            ngram_range=(1, 2),
            min_df=2,
            max_features=5_000,
        )),
        ("model", clone(estimator)),
    ])
    pipe.fit(train[text_col], train[target_col])
    pred = pipe.predict(test[text_col])

    scores = {
        "target": target_col,
        "text_col": text_col,
        "model": model_name,
        "train_start": train["date"].min().date(),
        "train_end": train["date"].max().date(),
        "test_start": test["date"].min().date(),
        "test_end": test["date"].max().date(),
        "accuracy": accuracy_score(test[target_col], pred),
        "macro_f1": f1_score(test[target_col], pred, labels=LABELS, average="macro", zero_division=0),
    }
    context_cols = [col for col in ["rate_decision", "next_rate_decision"] if col != target_col]
    predictions = test.loc[:, ["date", target_col, *context_cols]].assign(pred=pred)
    return scores, predictions, pipe


runs = []
predictions = {}
fitted = {}

for target_col in ["rate_decision", "next_rate_decision"]:
    for corpus in ["statement", "minutes"]:
        for text_version in ["clean", "masked"]:
            text_col = f"{corpus}_{'clean_text' if text_version == 'clean' else 'masked_text'}"
            for model_name, estimator in MODELS.items():
                scores, preds, pipe = fit_text_model(df, text_col, target_col, model_name, estimator)
                key = (target_col, corpus, text_version, model_name)
                runs.append({**scores, "corpus": corpus, "text_version": text_version})
                predictions[key] = preds
                fitted[key] = pipe

results = pd.DataFrame(runs)
results.head()

In [ ]:
(
    results
    .sort_values(["target", "text_version", "corpus", "macro_f1"], ascending=[True, True, True, False])
    .loc[:, ["target", "corpus", "text_version", "model", "accuracy", "macro_f1", "test_start", "test_end"]]
    .round(3)
)

Macro-F1 est le chiffre à regarder en premier. La classe `hold` est trop fréquente pour lire seulement l'accuracy.

In [ ]:
(
    results
    .query("model != 'majority'")
    .pivot_table(
        index=["target", "text_version", "corpus"],
        columns="model",
        values="macro_f1",
    )
    .round(3)
)

Dans ce split, les Minutes battent les Statements. Le signal reste modeste, mais il passe la baseline.

La version masquée sert surtout à voir si le modèle dépend de la phrase de décision.

In [ ]:
best = (
    results.query("target == 'next_rate_decision' and model != 'majority'")
    .sort_values("macro_f1", ascending=False)
    .iloc[0]
)
key = (best["target"], best["corpus"], best["text_version"], best["model"])

best.to_frame().T

In [ ]:
best_preds = predictions[key]
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(
    best_preds[best["target"]],
    best_preds["pred"],
    labels=LABELS,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("Best next-decision model");

Dernière lecture : les coefficients d'une logit sur Statements masqués. C'est descriptif, pas une évaluation.

In [ ]:
coef_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words=FED_STOP_WORDS,
        token_pattern=r"(?u)\b[a-z][a-z]+\b",
        ngram_range=(1, 2),
        min_df=2,
        max_features=5_000,
    )),
    ("model", LogisticRegression(max_iter=2_000, class_weight="balanced")),
])
coef_pipe.fit(df["statement_masked_text"], df["rate_decision"])

terms = coef_pipe.named_steps["tfidf"].get_feature_names_out()
model = coef_pipe.named_steps["model"]

coef_rows = []
for class_idx, label in enumerate(model.classes_):
    top = np.argsort(model.coef_[class_idx])[-15:][::-1]
    coef_rows.extend({"class": label, "term": terms[i], "coef": model.coef_[class_idx, i]} for i in top)

pd.DataFrame(coef_rows).round(3)

Les coefficients vont dans le sens attendu : `hike` charge sur inflation/prices/energy/supply, `cut` sur prospects/uncertainty/employment.